In [1]:
texts = [
    "Remain inside the box centred at (5, 5, 5) with sides 10 × 6 × 4.",
    "Remain inside the cube centred at (5, 5, 5) with sides 10 × 6 × 4. what is the center point, dimensions of the cube and the minimum and maximum points of this cube",
    "Stay inside the box from (−3, −3, 0) to (3, 3, 6).",
    "Stay inside the box centred at (0, 0, 0) with sides 20 × 20 × 10. what is the center point, dimensions of the cube and the minimum and maximum points of this cube",
    
]


In [ ]:
# for text in texts_to_check:
#     print(f"\n--- Testing Prompt: '{text}' ---")
#     response = chat(
#         model='llama3.1',
#         messages=[
#             {'role': 'system', 'content': 'You are a precise geospatial math assistant. When extracting shapes, map "box" to "cube". You MUST calculate each axis (X, Y, Z) separately in the scratchpad. Never mix them. sides 10 x 6 x 4 means side_x=10, side_y=6, side_z=4. Calculate min and max for each axis independently using center +/- (side / 2).'},
#             {'role': 'user', 'content': text}
#         ],
#         format=BarrierDetails.model_json_schema(),
#     )
    
#     barrier = BarrierDetails.model_validate_json(response.message.content)
#     print("shape_type:", barrier.shape_type)
#     print("min_point=", barrier.min_point, "max_point=", barrier.max_point)

In [12]:
from ollama import chat
from pydantic import BaseModel, Field

class ReasoningMixin:
    reasoning: str = Field(...,
        description="Explain the step-by-step thought process behind the provided values. Include key considerations and how they influenced the final decisions. MUST calculate X, Y, and Z axis independently using center +/- (side / 2).",
        repr=False, exclude=True
    )

class BarrierDetails(BaseModel, ReasoningMixin):
    # Type of shape being evaluated
    shape_type: str = Field(
        description="Type of shape. MUST be exactly 'cube' or 'sphere'. If the text says 'box', use 'cube'."
    )
    # Center coordinates
    center: list[float] = Field(
        description="The center of the object as [x, y, z]"
    )
    # Extents / dimensions
    side_lengths: list[float] = Field(
        description="Lengths of the sides as [x, y, z]"
    )
    # Calculated minimum bounding point
    min_point: list[float] = Field(
        description="Minimum bounding point [x, y, z]. From your reasoning, this is [min_x, min_y, min_z]."
    )
    # Calculated maximum bounding point
    max_point: list[float] = Field(
        description="Maximum bounding point [x, y, z]. From your reasoning, this is [max_x, max_y, max_z]."
    )

# Uncomment the loop below if you want to test all texts at once!
# for text in texts_to_check:
#     print(f"\n--- Testing Prompt: '{text}' ---")

# Select the index of the text you want to test (0, 1, 2, ...)
selected_index = 1
text = texts[selected_index]
print(f"\n--- Testing Prompt: '{text}' ---")

response = chat(
    model='llama3.1',
    messages=[
        {'role': 'system', 'content': 'You are a precise geospatial math assistant. When extracting shapes, map "box" to "cube". You MUST calculate each axis (X, Y, Z) separately in the reasoning layer. Never mix them. sides 10 x 6 x 4 means side_x=10, side_y=6, side_z=4. Calculate min and max for each axis independently using center +/- (side / 2).'},
        {'role': 'user', 'content': text}
    ],
    format=BarrierDetails.model_json_schema(),
    options={'temperature':0}
)

barrier = BarrierDetails.model_validate_json(response.message.content)
print("shape_type:", barrier.shape_type)
print("min_point=", barrier.min_point, "max_point=", barrier.max_point)



--- Testing Prompt: 'Remain inside the cube centred at (5, 5, 5) with sides 10 × 6 × 4. what is the center point, dimensions of the cube and the minimum and maximum points of this cube' ---
shape_type: cube
min_point= [-2.0, -1.0, -1.0] max_point= [12.0, 11.0, 7.0]


In [5]:
texts[3]

'Stay inside the box centred at (0, 0, 0) with sides 20 × 20 × 10. what is the center point, dimensions of the cube and the minimum and maximum points of this cube'

In [2]:
from ollama import chat
from pydantic import BaseModel, Field

# We no longer need the ReasoningMixin because the reasoning happens entirely in Pass 1.
class BarrierDetails(BaseModel):
    shape_type: str = Field(
        description="Type of shape. MUST be exactly 'cube' or 'sphere'. If the text says 'box', use 'cube'."
    )
    center: list[float] = Field(
        description="The center of the object as [x, y, z]"
    )
    side_lengths: list[float] = Field(
        description="Lengths of the sides as [x, y, z]"
    )
    min_point: list[float] = Field(
        description="Minimum bounding point [min_x, min_y, min_z]."
    )
    max_point: list[float] = Field(
        description="Maximum bounding point [max_x, max_y, max_z]."
    )

# The text input
text = texts[3]
print(f"\n--- Testing Prompt: '{text}' ---")

# ==========================================
# PASS 1: Unstructured Reasoning & Math
# ==========================================
# Let the model "think" freely in plain text to calculate the boundaries.
pass1_system = """You are a precise geospatial math assistant. 
1. Extract the shape (map 'box' to 'cube'), center coordinates, and side lengths from the user text. 
2. Calculate the min and max points for the X, Y, and Z axes separately. 
   Formula: min = center - (side / 2) | max = center + (side / 2).
Show your step-by-step math clearly."""

print("\nExecuting Pass 1 (Unstructured Math)...")
response_1 = chat(
    model='llama3.1',
    messages=[
        {'role': 'system', 'content': pass1_system},
        {'role': 'user', 'content': text}
    ],
    options={'temperature': 0}
)
unstructured_math = response_1.message.content
# print("\n--- Pass 1 Output ---\n", unstructured_math)

# ==========================================
# PASS 2: Structured JSON Extraction
# ==========================================
# Feed the completed math from Pass 1 into a strictly constrained JSON formatter.
pass2_system = "You are a data formatting engine. Extract the final calculated parameters from the provided text into the requested JSON schema."

print("\nExecuting Pass 2 (Structured JSON)...")
response_2 = chat(
    model='llama3.1',
    messages=[
        {'role': 'system', 'content': pass2_system},
        {'role': 'user', 'content': unstructured_math}
    ],
    format=BarrierDetails.model_json_schema(),
    options={'temperature': 0}
)

# Validate the final output with Pydantic
barrier = BarrierDetails.model_validate_json(response_2.message.content)

print("\n--- Final Structured Output ---")
print("Shape Type:", barrier.shape_type)
print("Center:", barrier.center)
print("Sides:", barrier.side_lengths)
print("Min Point:", barrier.min_point)
print("Max Point:", barrier.max_point)


--- Testing Prompt: 'Stay inside the box centred at (0, 0, 0) with sides 20 × 20 × 10. what is the center point, dimensions of the cube and the minimum and maximum points of this cube' ---

Executing Pass 1 (Unstructured Math)...

Executing Pass 2 (Structured JSON)...

--- Final Structured Output ---
Shape Type: cube
Center: [0.0, 0.0, 0.0]
Sides: [20.0, 20.0, 10.0]
Min Point: [-10.0, -10.0, -5.0]
Max Point: [10.0, 10.0, 5.0]


In [10]:
print(barrier)

shape_type='cube' center=[5.0, 5.0, 5.0] side_lengths=[10.0, 6.0, 4.0] min_point=[3.0, -0.5, 2.5] max_point=[7.0, 11.0, 7.5]


In [ ]:
# Testing the same text but without structured Pydantic output constraints
raw_response = chat(
    model='llama3.1',
    messages=[
        {'role': 'user', 'content': texts[3]} # text variable comes from the cell above
    ]
)

# print("--- Raw Unstructured Output ---")
print(raw_response.message.content)

barrier = BarrierDetails.model_validate_json(raw_response.message.content)
print(barrier)


Since you want to stay inside a box centered at (0, 0, 0), I assume you meant a "cube" instead of a "box", as boxes are typically rectangular in shape.

Here's how to calculate the center point, dimensions, and minimum and maximum points of this cube:

**Center Point:**
Since the cube is centered at (0, 0, 0), its center point is simply:
(0, 0, 0)

**Dimensions:**
You mentioned the sides are 20 × 20 × 10. This means the dimensions of the cube are:
Length: 20 units
Width: 20 units
Height: 10 units

**Minimum and Maximum Points:**

To find the minimum and maximum points of this cube, we need to determine the coordinates where each side intersects with the axes.

Let's consider each axis separately:

1. **X-axis ( Length ):**
The minimum point on the X-axis is at x = -10 (half of the length), and the maximum point is at x = 10.
2. **Y-axis ( Width ):**
The minimum point on the Y-axis is at y = -10 (half of the width), and the maximum point is at y = 10.
3. **Z-axis ( Height ):**
The minim

In [12]:
from ollama import chat
from pydantic import BaseModel


class control_barrier(BaseModel):
    type: str
    dim: int
    length: int

class contbf(control_barrier):
    cbf: list[control_barrier]


response = chat(
  model='llama3.1',
  messages=[{'role': 'user', 'content': 'Stay inside the square centred at (0, 0) with side 10.'}],
  format=contbf.model_json_schema(),
)

# bar = contbf.model_validate_json(response.message.content)
# print(bar)
print(response)

model='llama3.1' created_at='2026-04-15T11:23:59.3102242Z' done=True done_reason='stop' total_duration=10768861200 load_duration=198755300 prompt_eval_count=28 prompt_eval_duration=1698517700 eval_count=31 eval_duration=7946233900 message=Message(role='assistant', content='{ "type": "circle", "dim": 5, "length": 2, "cbf": [] }', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [4]:
from ollama import chat
from pydantic import BaseModel

class Pet(BaseModel):
  name: str
  animal: str
  age: int
  color: str | None
  favorite_toy: str | None

class PetList(BaseModel):
  pets: list[Pet]

response = chat(
  model='llama3.1',
  messages=[{'role': 'user', 'content': 'I have two cats named Luna and Loki, ages 4 and 10. I also have a dog named Teddy aged 25'}],
  format=PetList.model_json_schema(),
)

pets = PetList.model_validate_json(response.message.content)
print(pets)

pets=[Pet(name='Luna', animal='cat', age=4, color='', favorite_toy=''), Pet(name='Loki', animal='cat', age=10, color='', favorite_toy=''), Pet(name='Teddy', animal='dog', age=25, color='', favorite_toy='')]


In [10]:
pets.pets[2].name

'Teddy'

In [2]:
response = chat(
    model='llama3.1',
    messages=[{'role':'user','content':'How are you!'}],
)

print(response.message.content)

I'm just a computer program, so I don't have feelings or emotions like humans do. However, I'm functioning properly and ready to help with any questions or tasks you may have! How can I assist you today?
